<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CKIP-BERT Multi-Task Fine-Tuning

This notebook fine-tunes one `ckiplab/bert-base-chinese` model per fold with the existing shared-backbone and four-MLP-head architecture.

Key properties:

- External inference input remains `id,data`.
- Submission output remains `id,promise_status,verification_timeline,evidence_status,evidence_quality`.
- The masked BCE/CE objective and competition task weights are retained.
- Five folds are used both for reliable OOF evaluation and lower-variance probability ensembling.
- Each fold saves the epoch with the highest validation competition macro-F1, rather than the last epoch.
- Existing synthetic `Misleading` rows train T4 only and cannot alter T1-T3.


In [1]:
# Install dependencies in Colab, then restart the runtime if requested.
# !pip install -q transformers torch pandas numpy scikit-learn tqdm huggingface_hub


In [2]:
import gc
import json
import math
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import HfApi, notebook_login
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore")


In [17]:
# ==========================================
# 0. Configuration
# ==========================================

MODEL_NAME = "ckiplab/bert-base-chinese"
FOLDS = [1, 2, 3, 4, 5]
SEED = 42

MAX_LEN = 512
HEAD_RATIO = 0.25
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
MIN_SCORE_IMPROVEMENT = 1e-4

BACKBONE_LR = 1.5e-5
HEAD_LR = 7.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

SYNTHETIC_ID_MIN = 90000
SYNTHETIC_T4_SAMPLE_WEIGHT = 0.35
MAX_CLASS_WEIGHT = 5.0

TASK_WEIGHTS = {
    "t1": 0.20,
    "t2": 0.15,
    "t3": 0.30,
    "t4": 0.35,
}

ID_COLUMN = "id"
TEXT_COLUMN = "data"
TARGET_COLUMNS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

TASK_CLASSES = {
    "t1": ["No", "Yes"],
    "t2": ["already", "within_2_years", "between_2_and_5_years", "longer_than_5_years"],
    "t3": ["No", "Yes"],
    "t4": ["Clear", "Not Clear", "Misleading"],
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "app" / "data" / "clean_data").exists():
        PROJECT_ROOT = candidate
        break

LOCAL_DATA_DIR = PROJECT_ROOT / "app" / "data" / "clean_data"
RAW_BASE_URL = "https://raw.githubusercontent.com/Heng1222/VeriPromiseESG_2026_TEAM_9906/feat-model-train/app/data/clean_data/"

OUTPUT_DIR = Path("mtl_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OOF_PROBABILITY_CSV = OUTPUT_DIR / "mtl_oof_probabilities.csv"
OOF_PREDICTION_CSV = OUTPUT_DIR / "mtl_oof_predictions.csv"
THRESHOLD_JSON = OUTPUT_DIR / "mtl_thresholds.json"
INFERENCE_CONFIG_JSON = OUTPUT_DIR / "mtl_inference_config.json"
TOKENIZER_DIR = OUTPUT_DIR / "tokenizer"

RUN_HF_UPLOAD = True
HF_MTL_REPO_ID = "maxbeettww/VeriPromise_ESG_2026_9906"
HF_PRIVATE_REPO = False
HF_COMMIT_MESSAGE = "Upload CKIP-BERT MTL artifacts"

print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")


Device: cuda
Project root: /home/public/zhengheng/VeriPromiseESG_2026_TEAM_9906


In [4]:
# ==========================================
# 1. Data loading and tokenization
# ==========================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def normalize_value(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    if not value or value.upper() == "N/A":
        return None
    if value == "more_than_5_years":
        return "longer_than_5_years"
    return value


def read_fold_csv(fold, split):
    filename = f"{split}_fold_{fold}.csv"
    local_path = LOCAL_DATA_DIR / filename
    if local_path.exists():
        return pd.read_csv(local_path)
    return pd.read_csv(f"{RAW_BASE_URL}{filename}")


def validate_training_frame(df, name):
    required = [ID_COLUMN, TEXT_COLUMN] + TARGET_COLUMNS
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")
    if df[ID_COLUMN].duplicated().any():
        raise ValueError(f"{name} contains duplicated ids.")


def tokenize_head_tail(text, tokenizer, max_len=MAX_LEN, head_ratio=HEAD_RATIO):
    body_ids = tokenizer.encode(
        f"文本：{str(text)}",
        add_special_tokens=False,
        verbose=False,
    )
    max_body_len = max_len - 2
    if len(body_ids) > max_body_len:
        head_len = int(max_body_len * head_ratio)
        tail_len = max_body_len - head_len
        body_ids = body_ids[:head_len] + body_ids[-tail_len:]

    input_ids = [tokenizer.cls_token_id] + body_ids + [tokenizer.sep_token_id]
    attention_mask = [1] * len(input_ids)
    pad_len = max_len - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len
    return input_ids, attention_mask


class ESGMTLDataset(Dataset):
    T1_MAP = {"No": 0, "Yes": 1}
    T2_MAP = {
        "already": 0,
        "within_2_years": 1,
        "between_2_and_5_years": 2,
        "longer_than_5_years": 3,
        "more_than_5_years": 3,
    }
    T3_MAP = {"No": 0, "Yes": 1}
    T4_MAP = {"Clear": 0, "Not Clear": 1, "Misleading": 2}

    def __init__(self, dataframe, tokenizer, max_len=MAX_LEN, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        input_ids, attention_mask = tokenize_head_tail(
            row[TEXT_COLUMN],
            tokenizer=self.tokenizer,
            max_len=self.max_len,
        )
        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "row_index": torch.tensor(index, dtype=torch.long),
        }
        if self.is_test:
            return item

        is_synthetic = int(pd.to_numeric(row[ID_COLUMN], errors="coerce") >= SYNTHETIC_ID_MIN)
        t1_value = normalize_value(row.get("promise_status"))
        t2_value = normalize_value(row.get("verification_timeline"))
        t3_value = normalize_value(row.get("evidence_status"))
        t4_value = normalize_value(row.get("evidence_quality"))

        item.update(
            {
                "t1_label": torch.tensor(self.T1_MAP.get(t1_value, -1), dtype=torch.float),
                "t2_label": torch.tensor(self.T2_MAP.get(t2_value, -1), dtype=torch.long),
                "t3_label": torch.tensor(self.T3_MAP.get(t3_value, -1), dtype=torch.float),
                "t4_label": torch.tensor(self.T4_MAP.get(t4_value, -1), dtype=torch.long),
                "is_synthetic": torch.tensor(is_synthetic, dtype=torch.bool),
            }
        )
        return item


In [5]:
# ==========================================
# 2. Existing shared CKIP-BERT + four MLP heads
# ==========================================

class ESGUnifiedMTLModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        self.multi_sample_dropouts = nn.ModuleList(
            [nn.Dropout(probability) for probability in [0.1, 0.2, 0.3, 0.4, 0.5]]
        )
        self.t1_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
        )
        self.t3_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
        )
        self.t2_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 4),
        )
        self.t4_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 3),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]

        # Preserve the existing heads while making the intended multi-sample dropout active.
        t1_logits = torch.stack(
            [self.t1_head(dropout(cls_output)).squeeze(-1) for dropout in self.multi_sample_dropouts]
        ).mean(dim=0)
        t3_logits = torch.stack(
            [self.t3_head(dropout(cls_output)).squeeze(-1) for dropout in self.multi_sample_dropouts]
        ).mean(dim=0)
        t2_logits = self.t2_head(cls_output)
        t4_logits = self.t4_head(cls_output)
        return t1_logits, t2_logits, t3_logits, t4_logits


def sqrt_class_weights(counts, max_weight=MAX_CLASS_WEIGHT):
    counts = np.asarray(counts, dtype=float)
    largest = counts.max()
    weights = np.sqrt(largest / np.maximum(counts, 1e-8))
    return np.minimum(weights, max_weight)


def compute_fold_class_weights(train_df):
    work = train_df.copy()
    for column in TARGET_COLUMNS:
        work[column] = work[column].apply(normalize_value)
    synthetic = pd.to_numeric(work[ID_COLUMN], errors="coerce") >= SYNTHETIC_ID_MIN

    t1_counts = [
        int(((work["promise_status"] == label) & ~synthetic).sum())
        for label in TASK_CLASSES["t1"]
    ]
    t2_counts = [
        int(((work["verification_timeline"] == label) & ~synthetic).sum())
        for label in TASK_CLASSES["t2"]
    ]
    t3_counts = [
        int(((work["evidence_status"] == label) & ~synthetic).sum())
        for label in TASK_CLASSES["t3"]
    ]
    t4_counts = []
    for label in TASK_CLASSES["t4"]:
        real_count = int(((work["evidence_quality"] == label) & ~synthetic).sum())
        synthetic_count = int(((work["evidence_quality"] == label) & synthetic).sum())
        t4_counts.append(real_count + SYNTHETIC_T4_SAMPLE_WEIGHT * synthetic_count)

    return {
        "t1": torch.tensor(sqrt_class_weights(t1_counts), dtype=torch.float, device=DEVICE),
        "t2": torch.tensor(sqrt_class_weights(t2_counts), dtype=torch.float, device=DEVICE),
        "t3": torch.tensor(sqrt_class_weights(t3_counts), dtype=torch.float, device=DEVICE),
        "t4": torch.tensor(sqrt_class_weights(t4_counts), dtype=torch.float, device=DEVICE),
    }


def weighted_mean(losses, weights):
    return (losses * weights).sum() / weights.sum().clamp_min(1e-8)


def calculate_mtl_loss(predictions, batch, class_weights):
    t1_logits, t2_logits, t3_logits, t4_logits = predictions
    t1_labels = batch["t1_label"].to(DEVICE)
    t2_labels = batch["t2_label"].to(DEVICE)
    t3_labels = batch["t3_label"].to(DEVICE)
    t4_labels = batch["t4_label"].to(DEVICE)
    is_synthetic = batch["is_synthetic"].to(DEVICE)

    zero = t1_logits.sum() * 0.0
    losses = {"t1": zero, "t2": zero, "t3": zero, "t4": zero}

    # Synthetic Misleading rows are allowed to teach T4 only.
    real_mask = ~is_synthetic
    t1_valid = real_mask & (t1_labels >= 0)
    if t1_valid.any():
        element_loss = nn.functional.binary_cross_entropy_with_logits(
            t1_logits[t1_valid],
            t1_labels[t1_valid],
            reduction="none",
        )
        sample_weight = class_weights["t1"][t1_labels[t1_valid].long()]
        losses["t1"] = weighted_mean(element_loss, sample_weight)

    t2_valid = real_mask & (t1_labels == 1) & (t2_labels >= 0)
    if t2_valid.any():
        losses["t2"] = nn.functional.cross_entropy(
            t2_logits[t2_valid],
            t2_labels[t2_valid],
            weight=class_weights["t2"],
        )

    t3_valid = real_mask & (t1_labels == 1) & (t3_labels >= 0)
    if t3_valid.any():
        element_loss = nn.functional.binary_cross_entropy_with_logits(
            t3_logits[t3_valid],
            t3_labels[t3_valid],
            reduction="none",
        )
        sample_weight = class_weights["t3"][t3_labels[t3_valid].long()]
        losses["t3"] = weighted_mean(element_loss, sample_weight)

    t4_valid = (t1_labels == 1) & (t3_labels == 1) & (t4_labels >= 0)
    if t4_valid.any():
        element_loss = nn.functional.cross_entropy(
            t4_logits[t4_valid],
            t4_labels[t4_valid],
            weight=class_weights["t4"],
            reduction="none",
        )
        sample_weight = torch.where(
            is_synthetic[t4_valid],
            torch.full_like(element_loss, SYNTHETIC_T4_SAMPLE_WEIGHT),
            torch.ones_like(element_loss),
        )
        losses["t4"] = weighted_mean(element_loss, sample_weight)

    total_loss = sum(TASK_WEIGHTS[task] * loss for task, loss in losses.items())
    return total_loss, losses


In [6]:
# ==========================================
# 3. Probability routing and competition metrics
# ==========================================

def sigmoid_numpy(values):
    values = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-values))


def softmax_numpy(values):
    shifted = values - values.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def predict_probabilities(model, data_loader):
    model.eval()
    output = {task: [] for task in ["t1", "t2", "t3", "t4"]}
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(input_ids, attention_mask)
            output["t1"].append(sigmoid_numpy(logits[0].float().cpu().numpy()))
            output["t2"].append(softmax_numpy(logits[1].float().cpu().numpy()))
            output["t3"].append(sigmoid_numpy(logits[2].float().cpu().numpy()))
            output["t4"].append(softmax_numpy(logits[3].float().cpu().numpy()))
    return {task: np.concatenate(parts, axis=0) for task, parts in output.items()}


def probabilities_to_frame(df, probabilities):
    output = pd.DataFrame({ID_COLUMN: df[ID_COLUMN].values})
    output["t1__Yes"] = probabilities["t1"]
    output["t1__No"] = 1.0 - probabilities["t1"]
    output["t3__Yes"] = probabilities["t3"]
    output["t3__No"] = 1.0 - probabilities["t3"]
    for index, label in enumerate(TASK_CLASSES["t2"]):
        output[f"t2__{label}"] = probabilities["t2"][:, index]
    for index, label in enumerate(TASK_CLASSES["t4"]):
        output[f"t4__{label}"] = probabilities["t4"][:, index]
    return output


def argmax_label(row, task):
    labels = TASK_CLASSES[task]
    values = [row[f"{task}__{label}"] for label in labels]
    return labels[int(np.argmax(values))]


def route_predictions(probability_df, t1_threshold=0.5, t3_threshold=0.5):
    results = []
    for row_index, row in probability_df.iterrows():
        source_id = probability_df.at[row_index, ID_COLUMN]
        t1_prediction = "Yes" if row["t1__Yes"] >= t1_threshold else "No"
        if t1_prediction == "No":
            results.append(
                {
                    ID_COLUMN: source_id,
                    "promise_status": "No",
                    "verification_timeline": "N/A",
                    "evidence_status": "N/A",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t2_prediction = argmax_label(row, "t2")
        t3_prediction = "Yes" if row["t3__Yes"] >= t3_threshold else "No"
        if t3_prediction == "No":
            results.append(
                {
                    ID_COLUMN: source_id,
                    "promise_status": "Yes",
                    "verification_timeline": t2_prediction,
                    "evidence_status": "No",
                    "evidence_quality": "N/A",
                }
            )
            continue

        results.append(
            {
                ID_COLUMN: source_id,
                "promise_status": "Yes",
                "verification_timeline": t2_prediction,
                "evidence_status": "Yes",
                "evidence_quality": argmax_label(row, "t4"),
            }
        )
    return pd.DataFrame(results)[[ID_COLUMN] + TARGET_COLUMNS]


def strict_task_f1(true_df, prediction_df, column):
    merged = true_df[[ID_COLUMN, column]].merge(
        prediction_df[[ID_COLUMN, column]],
        on=ID_COLUMN,
        suffixes=("_true", "_pred"),
        validate="one_to_one",
    )
    y_true = merged[f"{column}_true"].apply(normalize_value)
    y_pred = merged[f"{column}_pred"].apply(normalize_value)
    valid = y_true.notna()
    task = {
        "promise_status": "t1",
        "verification_timeline": "t2",
        "evidence_status": "t3",
        "evidence_quality": "t4",
    }[column]
    return f1_score(
        y_true[valid],
        y_pred[valid].fillna("N/A"),
        labels=TASK_CLASSES[task],
        average="macro",
        zero_division=0,
    )


def evaluate_submission(true_df, prediction_df):
    scores = {
        column: strict_task_f1(true_df, prediction_df, column)
        for column in TARGET_COLUMNS
    }
    competition_score = (
        TASK_WEIGHTS["t1"] * scores["promise_status"]
        + TASK_WEIGHTS["t2"] * scores["verification_timeline"]
        + TASK_WEIGHTS["t3"] * scores["evidence_status"]
        + TASK_WEIGHTS["t4"] * scores["evidence_quality"]
    )
    rows = [
        {"task": column, "macro_f1": score}
        for column, score in scores.items()
    ]
    rows.append({"task": "competition", "macro_f1": competition_score})
    return pd.DataFrame(rows)


def print_reports(true_df, prediction_df):
    for column in TARGET_COLUMNS:
        task = {
            "promise_status": "t1",
            "verification_timeline": "t2",
            "evidence_status": "t3",
            "evidence_quality": "t4",
        }[column]
        merged = true_df[[ID_COLUMN, column]].merge(
            prediction_df[[ID_COLUMN, column]],
            on=ID_COLUMN,
            suffixes=("_true", "_pred"),
        )
        y_true = merged[f"{column}_true"].apply(normalize_value)
        y_pred = merged[f"{column}_pred"].apply(normalize_value)
        valid = y_true.notna()
        labels = TASK_CLASSES[task]
        print(f"\n=== {column} ===")
        print(classification_report(
            y_true[valid],
            y_pred[valid].fillna("N/A"),
            labels=labels,
            zero_division=0,
        ))
        print(pd.DataFrame(
            confusion_matrix(
                y_true[valid],
                y_pred[valid].fillna("N/A"),
                labels=labels,
            ),
            index=labels,
            columns=labels,
        ))


def tune_routing_thresholds(true_df, probability_df):
    best = {
        "score": -1.0,
        "t1_threshold": 0.5,
        "t3_threshold": 0.5,
    }
    grid = np.round(np.arange(0.20, 0.801, 0.01), 2)
    for t1_threshold in grid:
        for t3_threshold in grid:
            prediction_df = route_predictions(
                probability_df,
                t1_threshold=t1_threshold,
                t3_threshold=t3_threshold,
            )
            metrics = evaluate_submission(true_df, prediction_df)
            score = float(metrics.loc[metrics["task"] == "competition", "macro_f1"].iloc[0])
            if score > best["score"]:
                best = {
                    "score": score,
                    "objective": "competition_macro_f1",
                    "t1_threshold": float(t1_threshold),
                    "t3_threshold": float(t3_threshold),
                }
    return best


In [7]:
# ==========================================
# 4. Training helpers
# ==========================================

def create_optimizer(model):
    no_decay = ["bias", "LayerNorm.weight"]
    parameter_groups = []
    for module, learning_rate in [
        (model.backbone, BACKBONE_LR),
        (nn.ModuleList([model.t1_head, model.t2_head, model.t3_head, model.t4_head]), HEAD_LR),
    ]:
        named_parameters = list(module.named_parameters())
        parameter_groups.extend(
            [
                {
                    "params": [
                        parameter
                        for name, parameter in named_parameters
                        if not any(token in name for token in no_decay)
                    ],
                    "lr": learning_rate,
                    "weight_decay": WEIGHT_DECAY,
                },
                {
                    "params": [
                        parameter
                        for name, parameter in named_parameters
                        if any(token in name for token in no_decay)
                    ],
                    "lr": learning_rate,
                    "weight_decay": 0.0,
                },
            ]
        )
    return torch.optim.AdamW(parameter_groups)


def save_checkpoint(model, path, fold, epoch, metrics):
    torch.save(
        {
            "artifact_version": 2,
            "model_name": MODEL_NAME,
            "max_len": MAX_LEN,
            "head_ratio": HEAD_RATIO,
            "fold": fold,
            "best_epoch": epoch,
            "metrics": metrics,
            "model_state_dict": model.state_dict(),
        },
        path,
    )


def load_checkpoint(model, path):
    checkpoint = torch.load(path, map_location=DEVICE)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict)
    return checkpoint


def train_one_fold(fold, tokenizer):
    set_seed(SEED + fold)
    train_df = read_fold_csv(fold, "train")
    val_df = read_fold_csv(fold, "val")
    validate_training_frame(train_df, f"train_fold_{fold}")
    validate_training_frame(val_df, f"val_fold_{fold}")

    train_dataset = ESGMTLDataset(train_df, tokenizer)
    val_dataset = ESGMTLDataset(val_df, tokenizer)
    generator = torch.Generator().manual_seed(SEED + fold)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=USE_AMP,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=USE_AMP,
    )

    model = ESGUnifiedMTLModel(MODEL_NAME).to(DEVICE)
    class_weights = compute_fold_class_weights(train_df)
    print("Class weights:", {
        task: np.round(weights.detach().cpu().numpy(), 3).tolist()
        for task, weights in class_weights.items()
    })

    optimizer = create_optimizer(model)
    optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_steps = optimizer_steps_per_epoch * MAX_EPOCHS
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = fold_dir / "best_model.pth"

    best_score = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0
        task_loss_sums = {task: 0.0 for task in TASK_WEIGHTS}

        for step, batch in enumerate(tqdm(train_loader, desc=f"Fold {fold} epoch {epoch}"), start=1):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                predictions = model(input_ids, attention_mask)
                loss, task_losses = calculate_mtl_loss(predictions, batch, class_weights)
                scaled_loss = loss / GRAD_ACCUM_STEPS

            scaler.scale(scaled_loss).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += float(loss.detach().cpu())
            for task in task_loss_sums:
                task_loss_sums[task] += float(task_losses[task].detach().cpu())

        val_probabilities = predict_probabilities(model, val_loader)
        val_probability_df = probabilities_to_frame(val_df, val_probabilities)
        val_prediction_df = route_predictions(val_probability_df, 0.5, 0.5)
        val_metrics = evaluate_submission(
            val_df[[ID_COLUMN] + TARGET_COLUMNS],
            val_prediction_df,
        )
        val_score = float(
            val_metrics.loc[val_metrics["task"] == "competition", "macro_f1"].iloc[0]
        )
        row = {
            "fold": fold,
            "epoch": epoch,
            "train_loss": epoch_loss / len(train_loader),
            "competition_score": val_score,
        }
        for task in task_loss_sums:
            row[f"{task}_loss"] = task_loss_sums[task] / len(train_loader)
        for _, metric_row in val_metrics.iterrows():
            row[f"{metric_row['task']}_macro_f1"] = float(metric_row["macro_f1"])
        history.append(row)
        print(pd.DataFrame([row]).to_string(index=False))

        if val_score > best_score + MIN_SCORE_IMPROVEMENT:
            best_score = val_score
            best_epoch = epoch
            epochs_without_improvement = 0
            metrics_dict = {
                str(metric_row["task"]): float(metric_row["macro_f1"])
                for _, metric_row in val_metrics.iterrows()
            }
            save_checkpoint(
                model,
                checkpoint_path,
                fold=fold,
                epoch=epoch,
                metrics=metrics_dict,
            )
            print(f"Saved new best fold {fold} checkpoint: epoch={epoch}, score={val_score:.6f}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping fold {fold} after epoch {epoch}.")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(fold_dir / "training_history.csv", index=False)

    checkpoint = load_checkpoint(model, checkpoint_path)
    if int(checkpoint["best_epoch"]) != best_epoch:
        raise RuntimeError("Loaded checkpoint does not match the tracked best epoch.")
    val_probabilities = predict_probabilities(model, val_loader)
    val_probability_df = probabilities_to_frame(val_df, val_probabilities)
    val_prediction_df = route_predictions(val_probability_df, 0.5, 0.5)
    print(f"\nFold {fold} best epoch: {best_epoch}")
    display(evaluate_submission(val_df[[ID_COLUMN] + TARGET_COLUMNS], val_prediction_df))
    print_reports(val_df[[ID_COLUMN] + TARGET_COLUMNS], val_prediction_df)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        val_df[[ID_COLUMN] + TARGET_COLUMNS].copy(),
        val_probability_df,
        history_df,
    )


In [8]:
# ==========================================
# 5. Train exactly five folds and build OOF predictions
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.save_pretrained(TOKENIZER_DIR)

oof_truth_parts = []
oof_probability_parts = []
history_parts = []

for fold in FOLDS:
    print(f"\n{'=' * 60}\nTraining fold {fold}\n{'=' * 60}")
    fold_truth, fold_probabilities, fold_history = train_one_fold(fold, tokenizer)
    oof_truth_parts.append(fold_truth)
    oof_probability_parts.append(fold_probabilities)
    history_parts.append(fold_history)

oof_true_df = pd.concat(oof_truth_parts, ignore_index=True)
oof_probability_df = pd.concat(oof_probability_parts, ignore_index=True)
training_history_df = pd.concat(history_parts, ignore_index=True)

oof_probability_df.to_csv(OOF_PROBABILITY_CSV, index=False)
training_history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print(f"Saved OOF probabilities to {OOF_PROBABILITY_CSV}")



Training fold 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: {'t1': [2.0899999141693115, 1.0], 't2': [1.0, 4.5279998779296875, 1.1990000009536743, 1.38100004196167], 't3': [2.194000005722046, 1.0], 't4': [1.0, 2.2239999771118164, 4.681000232696533]}


Fold 1 epoch 1:   0%|          | 0/214 [00:00<?, ?it/s]

...

 fold  epoch  train_loss  competition_score  t1_loss  t2_loss  t3_loss  t4_loss  promise_status_macro_f1  verification_timeline_macro_f1  evidence_status_macro_f1  evidence_quality_macro_f1  competition_macro_f1
    5      5    0.458583           0.557263 0.353644 0.930382 0.381247 0.382638                 0.754625                        0.462476                   0.69015                   0.371206              0.557263


Fold 5 epoch 6:   0%|          | 0/214 [00:00<?, ?it/s]

 fold  epoch  train_loss  competition_score  t1_loss  t2_loss  t3_loss  t4_loss  promise_status_macro_f1  verification_timeline_macro_f1  evidence_status_macro_f1  evidence_quality_macro_f1  competition_macro_f1
    5      6    0.398946           0.563441 0.319261 0.855502 0.334074 0.304418                 0.762873                        0.482233                  0.708519                   0.359931              0.563441
Early stopping fold 5 after epoch 6.

Fold 5 best epoch: 4


,task,macro_f1
0,promise_status,0.745098
1,verification_timeline,0.455706
2,evidence_status,0.729807
3,evidence_quality,0.402205
4,competition,0.577090



=== promise_status ===
              precision    recall  f1-score   support

          No       0.76      0.45      0.57        75
         Yes       0.88      0.97      0.92       325

    accuracy                           0.87       400
   macro avg       0.82      0.71      0.75       400
weighted avg       0.86      0.87      0.86       400

     No  Yes
No   34   41
Yes  11  314

=== verification_timeline ===
                       precision    recall  f1-score   support

              already       0.62      0.71      0.66       143
       within_2_years       0.00      0.00      0.00         7
between_2_and_5_years       0.60      0.52      0.56       100
  longer_than_5_years       0.67      0.56      0.61        75

            micro avg       0.62      0.60      0.61       325
            macro avg       0.47      0.45      0.46       325
         weighted avg       0.61      0.60      0.60       325

                       already  within_2_years  between_2_and_5_years  \

In [9]:
# ==========================================
# 6. Tune routing thresholds and report final OOF metrics
# ==========================================

best_thresholds = tune_routing_thresholds(oof_true_df, oof_probability_df)
with open(THRESHOLD_JSON, "w", encoding="utf-8") as file:
    json.dump(best_thresholds, file, ensure_ascii=False, indent=2)
print("Best OOF routing thresholds:", best_thresholds)

oof_prediction_df = route_predictions(
    oof_probability_df,
    t1_threshold=best_thresholds["t1_threshold"],
    t3_threshold=best_thresholds["t3_threshold"],
)
oof_prediction_df.to_csv(OOF_PREDICTION_CSV, index=False)

oof_metrics = evaluate_submission(oof_true_df, oof_prediction_df)
display(oof_metrics)
print_reports(oof_true_df, oof_prediction_df)


Best OOF routing thresholds: {'score': 0.6011399224663667, 'objective': 'competition_macro_f1', 't1_threshold': 0.41, 't3_threshold': 0.35}


,task,macro_f1
0,promise_status,0.754363
1,verification_timeline,0.565558
2,evidence_status,0.724977
3,evidence_quality,0.422687
4,competition,0.601140



=== promise_status ===
              precision    recall  f1-score   support

          No       0.75      0.48      0.58       373
         Yes       0.89      0.96      0.92      1627

    accuracy                           0.87      2000
   macro avg       0.82      0.72      0.75      2000
weighted avg       0.86      0.87      0.86      2000

      No   Yes
No   179   194
Yes   61  1566

=== verification_timeline ===
                       precision    recall  f1-score   support

              already       0.66      0.61      0.63       718
       within_2_years       0.59      0.29      0.39        34
between_2_and_5_years       0.58      0.58      0.58       498
  longer_than_5_years       0.65      0.67      0.66       377

            micro avg       0.63      0.61      0.62      1627
            macro avg       0.62      0.54      0.57      1627
         weighted avg       0.63      0.61      0.62      1627

                       already  within_2_years  between_2_and_5_ye

In [12]:
from huggingface_hub import HfApi, login, create_repo
import os
from huggingface_hub import notebook_login

notebook_login()

In [18]:
# ==========================================
# 7. Save inference config and optionally upload artifacts
# ==========================================

def save_inference_config():
    thresholds = {"t1_threshold": 0.5, "t3_threshold": 0.5}
    if THRESHOLD_JSON.exists():
        with open(THRESHOLD_JSON, "r", encoding="utf-8") as file:
            thresholds.update(json.load(file))

    config = {
        "artifact_version": 2,
        "model_type": "ckip_bert_multi_task_mlp",
        "model_name": MODEL_NAME,
        "folds": FOLDS,
        "max_len": MAX_LEN,
        "head_ratio": HEAD_RATIO,
        "task_classes": TASK_CLASSES,
        "target_columns": TARGET_COLUMNS,
        "task_weights": TASK_WEIGHTS,
        "probability_ensemble": True,
        "thresholds": thresholds,
        "synthetic_policy": {
            "id_min": SYNTHETIC_ID_MIN,
            "t4_sample_weight": SYNTHETIC_T4_SAMPLE_WEIGHT,
            "tasks": ["t4"],
        },
        "artifact_layout": {
            "checkpoint": "fold_{fold}/best_model.pth",
            "tokenizer": "tokenizer/",
            "thresholds": "mtl_thresholds.json",
        },
    }
    with open(INFERENCE_CONFIG_JSON, "w", encoding="utf-8") as file:
        json.dump(config, file, ensure_ascii=False, indent=2)
    return config


def validate_artifacts():
    missing = []
    for fold in FOLDS:
        checkpoint = OUTPUT_DIR / f"fold_{fold}" / "best_model.pth"
        if not checkpoint.exists():
            missing.append(str(checkpoint))
    for path in [TOKENIZER_DIR, THRESHOLD_JSON, INFERENCE_CONFIG_JSON]:
        if not Path(path).exists():
            missing.append(str(path))
    if missing:
        raise FileNotFoundError("Missing MTL artifacts: " + ", ".join(missing))


def resolve_hf_repo_id(api, repo_id=None):
    if repo_id:
        return repo_id
    return f"{api.whoami()['name']}/maxbeettww/VeriPromise_ESG_2026_9906"


def push_artifacts_to_hf(repo_id=None, private=HF_PRIVATE_REPO):
    save_inference_config()
    validate_artifacts()
    api = HfApi()
    resolved_repo_id = resolve_hf_repo_id(api, repo_id)
    api.create_repo(
        repo_id=resolved_repo_id,
        repo_type="model",
        private=private,
        exist_ok=True,
    )
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=resolved_repo_id,
        repo_type="model",
        commit_message=HF_COMMIT_MESSAGE,
    )
    print(f"Uploaded CKIP-BERT MTL artifacts: https://huggingface.co/{resolved_repo_id}")
    return resolved_repo_id


inference_config = save_inference_config()
print(json.dumps(inference_config, ensure_ascii=False, indent=2))

if RUN_HF_UPLOAD:
    notebook_login()
    uploaded_repo_id = push_artifacts_to_hf(HF_MTL_REPO_ID)
else:
    print("RUN_HF_UPLOAD is False. Set it to True after training to upload mtl_outputs.")


{
  "artifact_version": 2,
  "model_type": "ckip_bert_multi_task_mlp",
  "model_name": "ckiplab/bert-base-chinese",
  "folds": [
    1,
    2,
    3,
    4,
    5
  ],
  "max_len": 512,
  "head_ratio": 0.25,
  "task_classes": {
    "t1": [
      "No",
      "Yes"
    ],
    "t2": [
      "already",
      "within_2_years",
      "between_2_and_5_years",
      "longer_than_5_years"
    ],
    "t3": [
      "No",
      "Yes"
    ],
    "t4": [
      "Clear",
      "Not Clear",
      "Misleading"
    ]
  },
  "target_columns": [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality"
  ],
  "task_weights": {
    "t1": 0.2,
    "t2": 0.15,
    "t3": 0.3,
    "t4": 0.35
  },
  "probability_ensemble": true,
  "thresholds": {
    "t1_threshold": 0.41,
    "t3_threshold": 0.35,
    "score": 0.6011399224663667,
    "objective": "competition_macro_f1"
  },
  "synthetic_policy": {
    "id_min": 90000,
    "t4_sample_weight": 0.35,
    "tasks": [
      "t4"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded CKIP-BERT MTL artifacts: https://huggingface.co/maxbeettww/VeriPromise_ESG_2026_9906
